# 08 — Hijack Execution Flow: PATH and DLL Search Order

| Field | Value |
|-------|-------|
| **MITRE ATT&CK ID** | T1574 — Hijack Execution Flow |
| **Tactic** | Persistence / Privilege Escalation |
| **Difficulty** | Intermediate |
| **Estimated Time** | 40 minutes |

## Threat Context: NotPetya — DLL Side-Loading

The NotPetya attack (2017) caused over $10 billion in damages worldwide. The malware spread through a compromised update mechanism of Ukrainian tax software (M.E.Doc), but its lateral movement relied on DLL hijacking and execution flow manipulation. NotPetya placed malicious DLLs in locations where legitimate Windows processes would load them, exploiting the Windows DLL search order.

**PATH hijacking** and **DLL search order hijacking** exploit a fundamental OS behavior: when a program requests a library or executable without specifying the full path, the OS searches directories in a defined order. An attacker who places a malicious file earlier in the search path can intercept execution.

## What You'll Understand After This

- How **PATH environment variable hijacking** allows attackers to substitute malicious executables
- The **Windows DLL search order** and how it creates hijacking opportunities
- How to **detect and prevent** execution flow hijacking through path hardening and integrity monitoring

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Environment setup and imports

import os
import sys
import tempfile
import shutil
import platform
import json

print("[+] Imports loaded successfully")
print(f"[i] Platform: {platform.system()}")
print("[i] This notebook demonstrates PATH hijacking concepts in a TEMP directory.")
print("[i] No system PATH or DLLs are actually modified.")

### Step 1 — Understand PATH Resolution

When you type a command like `python`, the OS searches each directory in the PATH variable, in order, until it finds a matching executable. An attacker who can write to a directory earlier in the PATH can hijack execution.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — PATH resolution demonstration

current_path = os.environ.get("PATH", "")
path_dirs = current_path.split(os.pathsep)

print(f"[+] Current PATH has {len(path_dirs)} directories:")
for i, d in enumerate(path_dirs[:10]):
    writable = os.access(d, os.W_OK) if os.path.exists(d) else False
    flag = " [WRITABLE]" if writable else ""
    print(f"  [{i:2d}] {d}{flag}")

if len(path_dirs) > 10:
    print(f"  ... and {len(path_dirs) - 10} more directories")

writable_dirs = [d for d in path_dirs if os.path.exists(d) and os.access(d, os.W_OK)]
print(f"\n[!] {len(writable_dirs)} directories in PATH are writable by current user")
print("[i] Writable PATH directories are potential hijack targets")

### Step 2 — Simulate PATH Hijacking

We create a temp directory, place a fake "executable" script in it, and show how prepending it to PATH would cause it to be found first.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — simulated PATH hijacking

hijack_dir = tempfile.mkdtemp(prefix="cyberlab_hijack_")
print(f"[+] Created hijack directory: {hijack_dir}")

# Create a fake script that simulates a hijacked command
if platform.system() == "Windows":
    fake_cmd_name = "notepad.bat"
    fake_cmd_content = '@echo [HIJACKED] This would have been the real notepad.exe\n@echo Attacker payload would execute here'
else:
    fake_cmd_name = "ls"  # Common command to hijack
    fake_cmd_content = '#!/bin/sh\necho "[HIJACKED] This would have been the real ls command"\necho "Attacker payload would execute here"'

fake_cmd_path = os.path.join(hijack_dir, fake_cmd_name)
with open(fake_cmd_path, "w") as f:
    f.write(fake_cmd_content)
os.chmod(fake_cmd_path, 0o755)

# Show the hijack concept (without actually modifying PATH)
print(f"[+] Created fake command: {fake_cmd_path}")
print(f"\n[i] PATH Hijack concept:")
print(f"    Original PATH: /usr/bin:/usr/local/bin:...")
print(f"    Hijacked PATH: {hijack_dir}:/usr/bin:/usr/local/bin:...")
print(f"    When user types '{fake_cmd_name}', the OS finds the FAKE version first")

# Simulate the search order
hijacked_path = [hijack_dir] + path_dirs[:5]
print(f"\n[+] Simulated search order resolution for '{fake_cmd_name}':")
path_hijack_demonstrated = False
for d in hijacked_path:
    candidate = os.path.join(d, fake_cmd_name)
    exists = os.path.exists(candidate)
    if exists and not path_hijack_demonstrated:
        print(f"  FOUND -> {candidate} [HIJACKED VERSION]")
        path_hijack_demonstrated = True
        break
    else:
        print(f"  skip    {d}/{fake_cmd_name} (not found)")

### Step 3 — Windows DLL Search Order

On Windows, DLL search order hijacking is even more impactful. When an application loads a DLL without a full path, Windows searches in a specific order. Placing a malicious DLL in the application's directory can hijack execution.

In [ ]:
# EDUCATIONAL PURPOSE ONLY — DLL search order documentation

dll_search_order = [
    {
        "priority": 1,
        "location": "Application directory",
        "hijack_risk": "HIGH — attacker drops DLL alongside .exe",
        "example": r"C:\Program Files\App\malicious.dll"
    },
    {
        "priority": 2,
        "location": "System directory (System32)",
        "hijack_risk": "LOW — requires admin to write",
        "example": r"C:\Windows\System32\malicious.dll"
    },
    {
        "priority": 3,
        "location": "16-bit system directory",
        "hijack_risk": "LOW — legacy path",
        "example": r"C:\Windows\System\malicious.dll"
    },
    {
        "priority": 4,
        "location": "Windows directory",
        "hijack_risk": "LOW — requires admin",
        "example": r"C:\Windows\malicious.dll"
    },
    {
        "priority": 5,
        "location": "Current working directory",
        "hijack_risk": "MEDIUM — CWD can be influenced",
        "example": r"C:\Users\user\Downloads\malicious.dll"
    },
    {
        "priority": 6,
        "location": "PATH directories",
        "hijack_risk": "MEDIUM — writable PATH dirs exploitable",
        "example": "Various PATH locations"
    },
]

print("[+] Windows DLL Search Order (SafeDllSearchMode enabled):")
print(f"{'#':<4} {'Location':<35} {'Hijack Risk'}")
print("-" * 75)
for entry in dll_search_order:
    print(f"{entry['priority']:<4} {entry['location']:<35} {entry['hijack_risk']}")

### Visualization — DLL Search Order Diagram

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis("off")
ax.set_title("Windows DLL Search Order — Hijacking Opportunities", fontsize=13, fontweight="bold")

risk_colors = {"HIGH": "#e74c3c", "MEDIUM": "#f39c12", "LOW": "#27ae60"}

y_pos = 7
for entry in dll_search_order:
    risk_key = entry["hijack_risk"].split(" ")[0]
    color = risk_colors.get(risk_key, "#95a5a6")

    # Priority number
    circle = plt.Circle((1, y_pos), 0.3, color=color, ec="white", linewidth=2)
    ax.add_patch(circle)
    ax.text(1, y_pos, str(entry["priority"]), ha="center", va="center",
            color="white", fontsize=12, fontweight="bold")

    # Location box
    rect = mpatches.FancyBboxPatch((2, y_pos - 0.35), 6.5, 0.7, boxstyle="round,pad=0.1",
                                   facecolor=color, edgecolor="white", alpha=0.2)
    ax.add_patch(rect)
    ax.text(2.2, y_pos + 0.05, entry["location"], fontsize=10, fontweight="bold",
            va="center")
    ax.text(5.5, y_pos + 0.05, entry["hijack_risk"], fontsize=8, va="center",
            color="#555", style="italic")

    # Arrow to next
    if entry["priority"] < 6:
        ax.annotate("", xy=(1, y_pos - 0.4), xytext=(1, y_pos - 0.7),
                    arrowprops=dict(arrowstyle="->", color="gray"))

    y_pos -= 1.1

# Legend
legend_items = [
    mpatches.Patch(color="#e74c3c", label="HIGH risk"),
    mpatches.Patch(color="#f39c12", label="MEDIUM risk"),
    mpatches.Patch(color="#27ae60", label="LOW risk"),
]
ax.legend(handles=legend_items, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()
print("[+] Visualization complete")

## Defender's Perspective — Indicators of Compromise

| Indicator | Type | Detection Method |
|-----------|------|------------------|
| DLL loaded from unexpected directory (not System32/app dir) | Behavioral | Sysmon Event ID 7 (Image Loaded), EDR DLL tracking |
| PATH environment variable modified by non-admin process | System | Environment variable monitoring, Sysmon Event 13 |
| Unsigned DLL loaded by signed application | File | Code signing verification, DLL whitelist enforcement |
| New file created in application installation directory | File | File integrity monitoring (OSSEC, Tripwire) |
| Process loading DLL from user-writable location | Behavioral | Application whitelisting, EDR process monitoring |

## Article Seed

**Title:** *Hijacking the Highway: How PATH and DLL Search Order Attacks Persist Undetected*

**Opening Paragraph:**
Every time your operating system searches for a program or library, it follows a predictable path — literally. Attackers exploit this predictability by placing malicious files where the OS will find them before the legitimate versions, hijacking execution flow without modifying a single byte of the original software.

**Section Headers:**
1. PATH Hijacking: The Unix and Windows Fundamentals
2. DLL Search Order Abuse: From NotPetya to Modern APTs
3. The DLL Side-Loading Ecosystem
4. Hardening Execution Flow: Secure PATH and DLL Policies

**Medium Tags:** `cybersecurity`, `dll-hijacking`, `persistence`, `mitre-attack`, `windows-security`

In [ ]:
# Self-check assertions

# 1. Verify PATH analysis was performed
assert len(path_dirs) > 0, "PATH should contain at least one directory"

# 2. Verify DLL search order was documented
assert len(dll_search_order) == 6, "Should document 6 DLL search order priorities"

# 3. Verify hijack simulation was set up
assert os.path.exists(fake_cmd_path), "Fake command should exist in hijack directory"
assert path_hijack_demonstrated, "PATH hijack should have been demonstrated"

# Cleanup
shutil.rmtree(hijack_dir, ignore_errors=True)
print("[+] All self-check assertions passed!")
print("[+] Temporary files cleaned up.")